# Week 8 — Simple Linear Regression (SE, RSE, t, p-value, พยากรณ์, Test MSE)

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week08/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ

## สูตรที่ใช้ทั้งสัปดาห์นี้ (X = [1, x] คือ 1 คอลัมน์ intercept + 1 คอลัมน์ตัวแปร)

| ปริมาณ | สูตร |
|---|---|
| ประมาณพารามิเตอร์ | β̂ = (XᵀX)⁻¹Xᵀy → โค้ดใช้ `np.linalg.solve(X.T@X, X.T@y)` |
| ส่วนเหลือ | e = y − Xβ̂ |
| RSS | Σeᵢ² |
| Residual Standard Error | **RSE = √(RSS / (n − 2))** |
| SE ของความชัน | **SE(β̂₁) = RSE / √(Σ(xᵢ − x̄)²)** |
| SE ของ intercept | SE(β̂₀) = RSE·√(1/n + x̄²/Σ(xᵢ − x̄)²) |
| t-statistic ของความชัน | **t = β̂₁ / SE(β̂₁)** |
| p-value (สองด้าน) | **p = 2·P(T > |t|)**, df = **n − 2** → `2*stats.t.sf(abs(t), df=n-2)` |

## กติกาที่ต้องระวัง

1. **"แถวที่ a ถึง b" = `.iloc[a:b+1]`** เช่น แถว 157–186 → `.iloc[157:187]`
2. **หารด้วย n − 2 ไม่ใช่ n** เพราะประมาณพารามิเตอร์ไป 2 ตัว (β₀, β₁) → df = n − 2
3. **residual ของแถวที่ r** ต้องทำให้**ค่าทำนายของแถวนั้น** ไม่ใช่ค่าเฉลี่ยของทั้งชุด
4. **โจทย์บางข้อกรองเฉพาะกลุ่มย่อย** (เช่น `channel == 'store'`, `season == 'Q1'`, `gender == 'F'`)
   → ต้องกรองก่อน แล้ว fit เฉพาะกลุ่มนั้น รวมถึง n ต้องนับจากกลุ่มนั้น
5. **แบ่ง train/test ตามตำแหน่งแถว** ไม่สุ่ม (ข้อ 8.9)
6. **โมเดล log-linear** (ข้อ 8.10) คำนวณบนสเกล ln แล้วต้อง **แปลงกลับด้วย `np.exp`**

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week08')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

ไฟล์ครบแล้ว: data\week08


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week08')
NEEDED = ['biz_campaigns.csv', 'biz_customers.csv']

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์  {list(df.columns)}')

biz_campaigns.csv        240 แถว x  7 คอลัมน์  ['tv', 'radio', 'social', 'email', 'season', 'sales', 'total_budget']
biz_customers.csv       1203 แถว x 13 คอลัมน์  ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member', 'web_visits', 'email_opens', 'basket_size', 'spend_k', 'orders_90d', 'churn', 'responded']


## ข้อ 8.1 — Standard Error ของความชัน SE(β̂₁)

ลำดับการคำนวณ (สำคัญ — ห้ามสลับ):
1. fit β̂ ด้วย Least Squares
2. หา RSS = Σeᵢ²
3. RSE = √(RSS / (n − 2))
4. **SE(β̂₁) = RSE / √(Σ(xᵢ − x̄)²)**

In [3]:
# --- ข้อ 8.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week08') / 'biz_customers.csv')   # ลูกค้า 1203 คน

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


x = cust['web_visits'].to_numpy(dtype=float)
y = cust['spend_k'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
r = rse(rss, n)
se_b1 = se_slope(x, r)
answer = se_b1

print('n =', n, '| beta0 =', round(b0, 4), '| beta1 =', round(b1, 6))
print('RSS =', round(rss, 4), '-> RSE = sqrt(RSS/(n-2)) =', round(r, 4))
print('sum((x - mean(x))^2) =', round(float(((x - x.mean()) ** 2).sum()), 2))
print('SE(beta1) =', answer)
print('(เทียบ t = beta1/SE =', round(b1 / se_b1, 2), '-> ความชันนัยสำคัญมาก)')
print('ค่าที่กรอก: 5 ตำแหน่ง ->', f'{answer:.5f}')

n = 1203 | beta0 = 2.4962 | beta1 = 0.11995
RSS = 2716.1227 -> RSE = sqrt(RSS/(n-2)) = 1.5038
sum((x - mean(x))^2) = 14773.99
SE(beta1) = 0.012372408897686626
(เทียบ t = beta1/SE = 9.69 -> ความชันนัยสำคัญมาก)
ค่าที่กรอก: 5 ตำแหน่ง -> 0.01237


## ข้อ 8.2 — Residual Standard Error (RSE)

- **RSE = √(RSS / (n − 2))** หาร n−2 เพราะมีพารามิเตอร์ 2 ตัว
- RSE คือ "ขนาดความคลาดเคลื่อนมาตรฐานของแบบจำลอง" มีหน่วยเดียวกับ y

In [4]:
# --- ข้อ 8.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week08') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


x = camp['social'].to_numpy(dtype=float)
y = camp['sales'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
answer = rse(rss, n)

print('n =', n, '| beta0 =', round(b0, 4), '| beta1 =', round(b1, 6))
print('RSS =', round(rss, 4))
print(f'RSE = sqrt(RSS / (n-2)) = sqrt({rss:.2f} / {n - 2}) =', round(answer, 6))
print('(หารด้วย n แทนจะได้', round(float(np.sqrt(rss / n)), 4), '- ผิด เพราะต้องใช้ df = n-2)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n = 240 | beta0 = 19.3206 | beta1 = 0.000813
RSS = 16342.0655
RSE = sqrt(RSS / (n-2)) = sqrt(16342.07 / 238) = 8.286383
(หารด้วย n แทนจะได้ 8.2518 - ผิด เพราะต้องใช้ df = n-2)
ค่าที่กรอก: 4 ตำแหน่ง -> 8.2864


## ข้อ 8.3 — Intercept ของโมเดลเฉพาะลูกค้าที่ซื้อหน้าร้าน

- กรอง `channel == 'store'` **ก่อน** แล้ว fit เฉพาะกลุ่มนั้น
- β̂₀ = ค่าทำนาย `spend_k` เมื่อ `tenure_months` = 0
- เป็นค่า**นอกช่วงข้อมูลจริง** (tenure ไม่มีใครเป็น 0) → ตีความอย่างระมัดระวัง

In [5]:
# --- ข้อ 8.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week08') / 'biz_customers.csv')   # ลูกค้า 1203 คน

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


store = cust[cust['channel'] == 'store']
x = store['tenure_months'].to_numpy(dtype=float)
y = store['spend_k'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
answer = b0

print('จำนวนลูกค้าที่ซื้อหน้าร้าน =', n, 'จากทั้งหมด', len(cust))
print('beta0 =', answer, '| beta1 =', round(b1, 6))
print('ช่วงของ tenure_months ในกลุ่มนี้ =', int(x.min()), 'ถึง', int(x.max()),
      '-> ไม่มีค่า 0 จึงเป็นการตีความนอกช่วงข้อมูล')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนลูกค้าที่ซื้อหน้าร้าน = 445 จากทั้งหมด 1203
beta0 = 3.069125582134775 | beta1 = 0.00323
ช่วงของ tenure_months ในกลุ่มนี้ = 1 ถึง 96 -> ไม่มีค่า 0 จึงเป็นการตีความนอกช่วงข้อมูล
ค่าที่กรอก: 4 ตำแหน่ง -> 3.0691


## ข้อ 8.4 — พยากรณ์ยอดขายของแคมเปญใหม่ (งบทีวี 185 พันบาท)

- fit เฉพาะแคมเปญ `season == 'Q1'`
- ทำนาย = **β̂₀ + β̂₁ · 185**
- นี่คือค่าทำนายของ **ค่าเฉลี่ย (conditional mean)** ไม่ใช่ค่าของแคมเปญรายตัว

In [6]:
# --- ข้อ 8.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week08') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


q1 = camp[camp['season'] == 'Q1']
x = q1['tv'].to_numpy(dtype=float)
y = q1['sales'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
tv_new = 185.0
answer = b0 + b1 * tv_new

print('จำนวนแคมเปญ Q1 =', n)
print('สมการ: yhat =', round(b0, 4), '+', round(b1, 6), '* tv')
print('ที่ tv =', tv_new, '-> yhat =', round(b0, 4), '+', round(b1, 6), '*', tv_new, '=', answer)
print('(ช่วง tv ของแคมเปญ Q1 =', round(float(x.min()), 1), 'ถึง', round(float(x.max()), 1), '-> 185 อยู่ในช่วง)')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

จำนวนแคมเปญ Q1 = 67
สมการ: yhat = 6.1603 + 0.086503 * tv
ที่ tv = 185.0 -> yhat = 6.1603 + 0.086503 * 185.0 = 22.16324864795358
(ช่วง tv ของแคมเปญ Q1 = 8.4 ถึง 292.8 -> 185 อยู่ในช่วง)
ค่าที่กรอก: 3 ตำแหน่ง -> 22.163


## ข้อ 8.5 — t-statistic ของความชัน (ทดสอบ H₀: β₁ = 0)

- **t = β̂₁ / SE(β̂₁)** โดย SE(β̂₁) ต้องคำนวณจาก RSE ของโมเดลนี้
- |t| มาก = ความชันต่างจาก 0 อย่างมีนัยสำคัญ

In [7]:
# --- ข้อ 8.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week08') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


x = camp['total_budget'].to_numpy(dtype=float)
y = camp['sales'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
r = rse(rss, n)
se_b1 = se_slope(x, r)
t = b1 / se_b1
answer = float(t)

print('n =', n, '| beta1 =', round(b1, 6))
print('RSE =', round(r, 4), '| SE(beta1) =', round(se_b1, 6))
print('t =', round(b1, 6), '/', round(se_b1, 6), '=', answer)
print('(t ขนาดใหญ่มาก = งบโฆษณารวมมีอิทธิพลต่อยอดขายอย่างชัดเจน)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

n = 240 | beta1 = 0.079001
RSE = 4.5624 | SE(beta1) = 0.003378
t = 0.079001 / 0.003378 = 23.390251157698923
(t ขนาดใหญ่มาก = งบโฆษณารวมมีอิทธิพลต่อยอดขายอย่างชัดเจน)
ค่าที่กรอก: 4 ตำแหน่ง -> 23.3903


## ข้อ 8.6 — p-value ของความชันแบบสองด้าน (n = 30)

- ข้อมูลนำร่องแถวที่ 157 ถึง 186 → `.iloc[157:187]` (30 แถว)
- **df = n − 2 = 28**
- p = 2·P(T > |t|) → `2 * stats.t.sf(abs(t), df=n-2)`

In [8]:
# --- ข้อ 8.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week08') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))

from scipy import stats

sub = camp.iloc[157:187]                  # แถว 157-186 รวมทั้งสองปลาย
x = sub['social'].to_numpy(dtype=float)
y = sub['sales'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
r = rse(rss, n)
se_b1 = se_slope(x, r)
t = b1 / se_b1
df = n - 2
p = 2 * stats.t.sf(abs(t), df=df)
answer = float(p)

print('จำนวนแถว =', len(sub), '(ควรเป็น 30) | df = n-2 =', df)
print('beta1 =', round(b1, 6), '| SE =', round(se_b1, 6), '| t =', round(float(t), 4))
print('p-value =', answer, '(p > 0.05 = ยังไม่ปฏิเสธ H0)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนแถว = 30 (ควรเป็น 30) | df = n-2 = 28
beta1 = -0.111615 | SE = 0.075422 | t = -1.4799
p-value = 0.15006969352865357 (p > 0.05 = ยังไม่ปฏิเสธ H0)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1501


## ข้อ 8.7 — RSS ของโมเดลเฉพาะลูกค้าเพศหญิง

- กรอง `gender == 'F'` (602 คน) แล้ว fit เฉพาะกลุ่ม
- RSS = Σ(yᵢ − ŷᵢ)² รวมทุกแถวของกลุ่มนั้น

In [9]:
# --- ข้อ 8.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week08') / 'biz_customers.csv')   # ลูกค้า 1203 คน

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


fem = cust[cust['gender'] == 'F']
x = fem['web_visits'].to_numpy(dtype=float)
y = fem['spend_k'].to_numpy(dtype=float)

b0, b1, resid, rss, n = slr(x, y)
answer = rss

print('จำนวนลูกค้าเพศหญิง =', n, '(โจทย์ระบุ 602)')
print('สมการ: yhat =', round(b0, 4), '+', round(b1, 6), '* web_visits')
print('RSS =', round(answer, 6), '| RSE =', round(rse(rss, n), 4))
print('(RSS ต่อคนเฉลี่ย =', round(answer / n, 4), '-> เทียบระหว่างเพศได้โดยใช้ค่านี้)')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

จำนวนลูกค้าเพศหญิง = 602 (โจทย์ระบุ 602)
สมการ: yhat = 2.6955 + 0.101944 * web_visits
RSS = 1317.817143 | RSE = 1.482
(RSS ต่อคนเฉลี่ย = 2.1891 -> เทียบระหว่างเพศได้โดยใช้ค่านี้)
ค่าที่กรอก: 2 ตำแหน่ง -> 1317.82


## ข้อ 8.8 — ส่วนเหลือ (residual) ของแถวที่ 233

- eᵢ = yᵢ − ŷᵢ โดย ŷᵢ คำนวณจาก **x ของแถว 233** ไม่ใช่ค่าเฉลี่ย
- ค่าติดลบ = โมเดลทำนาย**สูงกว่า**ค่าจริง

In [10]:
# --- ข้อ 8.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week08') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


x = camp['email'].to_numpy(dtype=float)
y = camp['sales'].to_numpy(dtype=float)
b0, b1, resid, rss, n = slr(x, y)

row = camp.iloc[233]                    # นับแถวแรกเป็นแถวที่ 0
yhat = b0 + b1 * float(row['email'])
answer = float(row['sales'] - yhat)

print('แถวที่ 233: email =', row['email'], ', sales จริง =', row['sales'])
print('yhat =', round(b0, 4), '+', round(b1, 6), '*', row['email'], '=', round(yhat, 4))
print('e = y - yhat =', answer, '(ติดลบ = ทำนายสูงกว่าจริง', round(yhat - float(row['sales']), 4), ')')
print('เทียบ residual ที่ได้จากอาร์เรย์ =', round(float(resid[233]), 6))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

แถวที่ 233: email = 16.2 , sales จริง = 10.11
yhat = 21.1719 + -0.118647 * 16.2 = 19.2498
e = y - yhat = -9.139794277228443 (ติดลบ = ทำนายสูงกว่าจริง 9.1398 )
เทียบ residual ที่ได้จากอาร์เรย์ = -9.139794
ค่าที่กรอก: 4 ตำแหน่ง -> -9.1398


## ข้อ 8.9 — Test MSE ของโมเดลที่ train บน 960 แถว

- train = 960 แถวแรก, test = 243 แถวที่เหลือ (แบ่งตามตำแหน่งแถว ไม่สุ่ม)
- Test MSE = ค่าเฉลี่ยของ (y − ŷ)² บนชุด test **เท่านั้น**

In [11]:
# --- ข้อ 8.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week08') / 'biz_customers.csv')   # ลูกค้า 1203 คน

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


train = cust.iloc[:960]
test = cust.iloc[960:]

xtr = train['income_k'].to_numpy(dtype=float)
ytr = train['spend_k'].to_numpy(dtype=float)
b0, b1, resid, rss, n = slr(xtr, ytr)

xte = test['income_k'].to_numpy(dtype=float)
yte = test['spend_k'].to_numpy(dtype=float)
pred = b0 + b1 * xte
mse = float(((yte - pred) ** 2).mean())
answer = mse

print('สมการที่ fit ด้วย train:', round(b0, 4), '+', round(b1, 6), '* income_k')
print('train MSE =', round(float(((ytr - (b0 + b1 * xtr)) ** 2).mean()), 6))
print('test  MSE =', answer, '| test =', len(test), 'แถว')
print('(test MSE สูงกว่า train MSE เล็กน้อย = ไม่ได้ overfit)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

สมการที่ fit ด้วย train: 1.9701 + 0.049126 * income_k
train MSE = 1.980009
test  MSE = 1.9545343439256009 | test = 243 แถว
(test MSE สูงกว่า train MSE เล็กน้อย = ไม่ได้ overfit)
ค่าที่กรอก: 4 ตำแหน่ง -> 1.9545


## ข้อ 8.10 — โมเดล log-linear แล้วแปลงกลับ

- y = **ln(`spend_k`)** ไม่ใช่ `spend_k` ตรง ๆ (เพราะข้อมูลเบ้ขวา)
- fit SLR ปกติ แล้วพยากรณ์ที่ `income_k` = 25
- ผลที่ได้คือค่า **บนสเกล ln** → ต้อง **`np.exp`** เพื่อกลับเป็นหน่วยพันบาท

In [12]:
# --- ข้อ 8.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week08') / 'biz_customers.csv')   # ลูกค้า 1203 คน

def slr(x, y):
    """Simple Linear Regression: คืน (beta0, beta1, residual, RSS, n)"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    resid = y - X @ beta
    return float(beta[0]), float(beta[1]), resid, float(resid @ resid), len(x)


def rse(rss, n):
    """Residual Standard Error = sqrt(RSS / (n - 2))"""
    return float(np.sqrt(rss / (n - 2)))


def se_slope(x, rse_val):
    """SE(beta1) = RSE / sqrt(sum((x - mean(x))^2))"""
    x = np.asarray(x, dtype=float)
    return float(rse_val / np.sqrt(((x - x.mean()) ** 2).sum()))


x = cust['income_k'].to_numpy(dtype=float)
y = np.log(cust['spend_k'].to_numpy(dtype=float))     # ln ของยอดซื้อ

b0, b1, resid, rss, n = slr(x, y)
pred_log = b0 + b1 * 25.0
answer = float(np.exp(pred_log))                      # แปลงกลับเป็นพันบาท

print('สมการบนสเกล ln: ln(yhat) =', round(b0, 6), '+', round(b1, 8), '* income_k')
print('ที่ income_k = 25 -> ln(yhat) =', round(pred_log, 6))
print('แปลงกลับด้วย exp ->', answer, 'พันบาท')
print('(เทียบค่าเฉลี่ยจริงของ spend_k =', round(float(cust['spend_k'].mean()), 4),
      '-> exp ของค่าพยากรณ์บนสเกล ln คือ "ค่ามัธยฐาน" ไม่ใช่ค่าเฉลี่ย)')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

สมการบนสเกล ln: ln(yhat) = 0.633305 + 0.01532107 * income_k
ที่ income_k = 25 -> ln(yhat) = 1.016332
แปลงกลับด้วย exp -> 2.7630416082087423 พันบาท
(เทียบค่าเฉลี่ยจริงของ spend_k = 3.4747 -> exp ของค่าพยากรณ์บนสเกล ln คือ "ค่ามัธยฐาน" ไม่ใช่ค่าเฉลี่ย)
ค่าที่กรอก: 3 ตำแหน่ง -> 2.763


## สรุปคำตอบทั้ง 10 ข้อ (Week 8)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)

| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 8.1 | SE(beta1) ของ spend_k ~ web_visits | **0.01237** | 5 ตำแหน่ง | 0.000015 |
| 8.2 | RSE ของ sales ~ social | **8.2864** | 4 ตำแหน่ง | 0.00015 |
| 8.3 | intercept ของ spend_k ~ tenure (ลูกค้าหน้าร้าน) | **3.0691** | 4 ตำแหน่ง | 0.00015 |
| 8.4 | พยากรณ์ sales ที่ tv=185 (fit เฉพาะ Q1) | **22.163** | 3 ตำแหน่ง | 0.0015 |
| 8.5 | t-statistic ของความชัน sales ~ total_budget | **23.3903** | 4 ตำแหน่ง | 0.00015 |
| 8.6 | p-value ของความชัน (30 แถว, df=28) | **0.1501** | 4 ตำแหน่ง | 0.00015 |
| 8.7 | RSS ของสตรี: spend_k ~ web_visits | **1317.82** | 2 ตำแหน่ง | 0.015 |
| 8.8 | residual ของแถวที่ 233 (sales ~ email) | **-9.1398** | 4 ตำแหน่ง | 0.00015 |
| 8.9 | Test MSE ของ spend_k ~ income_k (960/243) | **1.9545** | 4 ตำแหน่ง | 0.00015 |
| 8.10 | พยากรณ์ที่ income_k=25 จากโมเดล log-linear | **2.763** | 3 ตำแหน่ง | 0.0015 |

### จำสูตรไว้ใช้ต่อ

```
RSS          = Σ(y - ŷ)²
RSE          = √(RSS / (n - 2))
SE(β₁)       = RSE / √(Σ(xᵢ - x̄)²)
t            = β₁ / SE(β₁)
p (สองด้าน)  = 2 * P(T > |t|), df = n - 2
```

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน